In [ ]:
import os, sys
sys.path.append('..')

import time
import copy, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.nn.utils import spectral_norm
from sklearn.metrics.pairwise import pairwise_distances


In [2]:
class Sampler:
    def __init__(
        self, device='cuda',
    ):
        self.device = device
    
    def sample(self, size=5):
        pass

class TensorSampler(Sampler):
    def __init__(self, tensor, device='cuda'):
        super(TensorSampler, self).__init__(device)
        self.tensor = torch.clone(tensor).to(device)
        
    def sample(self, size=5):
        assert size <= self.tensor.shape[0]
        
        ind = torch.tensor(np.random.choice(np.arange(self.tensor.shape[0]), size=size, replace=False), device=self.device)
        return torch.clone(self.tensor[ind]).detach().to(self.device)


In [ ]:
DIM        = 100
SEED       = 42
EPSILON    = 0.1
BATCH_SIZE = 128
K          = 128
LR         = 1e-4
EPOCH      = 500

DAY_START  = 2
DAY_END    = 4
DAY_EVAL   = 3

LMC_STEPS     = 200
LMC_STEP_SIZE = 0.01

EVAL_EVERY = 2000

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DIM > 1


In [4]:
hiddens = {50: [256, 256, 256], 100 : [256, 256, 256], 1000: [2048, 1024, 512]}

In [5]:
hiddens[DIM]

[256, 256, 256]

In [6]:
torch.manual_seed(SEED)
np.random.seed(SEED)

EXP_NAME    = f'EOT_MSCI_DIM_{DIM}_EPS_{EPSILON}_SEED_{SEED}_DAY_{DAY_START}to{DAY_END}'
OUTPUT_PATH = f'../checkpoints/{EXP_NAME}'
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Experiment: {EXP_NAME}')
print(f'Device: {DEVICE}')


Experiment: EOT_MSCI_DIM_100_EPS_0.1_SEED_42_DAY_2to4
Device: cpu


In [7]:
data = {}
for day in [2, 3, 4, 7]:
    data[day] = np.load(f'../data/full_cite_pcas_{DIM}_day_{day}.npy')

eval_data  = data[DAY_EVAL]
start_data = data[DAY_START]
end_data   = data[DAY_END]

constant_scale = np.concatenate([start_data, end_data, eval_data]).std(axis=0).mean()
print(f'constant_scale = {constant_scale:.4f}')

start_data_scaled = start_data / constant_scale
end_data_scaled   = end_data   / constant_scale
eval_data_scaled  = eval_data  / constant_scale

eval_data_raw = torch.tensor(eval_data).float()

X_sampler = TensorSampler(torch.tensor(start_data_scaled).float(), device='cpu')
Y_sampler = TensorSampler(torch.tensor(end_data_scaled).float(),   device='cpu')

print(f'Source: {start_data_scaled.shape},  Target: {end_data_scaled.shape},  Eval: {eval_data.shape}')


constant_scale = 4.5619
Source: (6071, 100),  Target: (8485, 100),  Eval: (7643, 100)


In [ ]:
class EOTConfig:
    def __init__(self,
                 eps: float = 0.1,
                 batch_size: int = 2048,
                 device: str ="cpu",
                 K: int = 32,
                 epoch: int = 100,
                 lmc_steps: int = 100,
                 lmc_step_size: float = 0.003,
                 seed: int = 42,
                 grad_clip = 100000.0,
                 max_diff_exp_clip=100,
                 ema_momentum = 0.999
        ):
        self.device = device
        self.eps = eps
        self.batch_size = batch_size
        self.K = K
        self.epoch = epoch
        self.grad_clip = grad_clip
        self.max_diff_exp_clip = max_diff_exp_clip
        self.lmc_steps = lmc_steps
        self.lmc_step_size = lmc_step_size
        self.seed = seed
        self.ema_momentum = ema_momentum


class MLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hiddens,
        output_dim,
        activation_gen=lambda: nn.ReLU(),
        sn_iters=0
    ):

        def _SN(module):
            if sn_iters == 0:
                return module
            return spectral_norm(
                module, init=False, zero_bias=False, n_iters=sn_iters)

        assert isinstance(hiddens, list)
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hiddens = hiddens

        model = []
        prev_h = input_dim
        for h in hiddens:
            model.append(_SN(nn.Linear(prev_h, h)))
            model.append(activation_gen())
            prev_h = h
        model.append(_SN(nn.Linear(hiddens[-1], output_dim)))
        self.net = nn.Sequential(*model)

    def forward(self, x):
        batch_size = x.shape[0]
        x = x.view(batch_size, -1)
        return self.net(x).view(batch_size, self.output_dim)

In [ ]:
class EOTTrainer:
    def __init__(self, config, source_sampler, target_sampler, model_theta, model_phi, name):
        self.experiment_name = name

        self.config = config
        self.source_sampler = source_sampler
        self.target_sampler = target_sampler

        self.f_theta = model_theta
        self.f_phi = model_phi
        self.f_theta_ema = copy.deepcopy(self.f_theta).eval()
        for p in self.f_theta_ema.parameters(): p.requires_grad_(False)
        self.current_step = 0

    def ema_update(self, model, ema):
        m = self.config.ema_momentum
        with torch.no_grad():
            for p, pe in zip(model.parameters(), ema.parameters()):
                pe.mul_(m).add_(p, alpha=1-m)


    def compute_loss(self, x, y):
        cfg = self.config
        broad_shape = list(x.shape)
        broad_shape.insert(1, cfg.K)
        z = torch.randn(size=broad_shape, device=cfg.device)

        x_noisy = x[:, None, :] - math.sqrt(cfg.eps(self.current_step)) * z

        fphi_x = self.f_phi(x)
        ftheta_xnoisy = self.f_theta(x_noisy.reshape(-1, *x_noisy.shape[2:])).view(cfg.batch_size, cfg.K) / cfg.eps(self.current_step)
        ftheta_y = self.f_theta(y)

        diff_in_exp = ftheta_xnoisy - fphi_x
        diff_in_exp_truncated = torch.clamp(diff_in_exp, min=None, max=self.config.max_diff_exp_clip)
        exp_term = torch.exp(diff_in_exp_truncated.to(torch.float64))

        ftheta_y_mean = ftheta_y.mean()
        fphi_x_mean = fphi_x.mean()
        return (fphi_x_mean + exp_term.mean()) * cfg.eps(self.current_step) - ftheta_y_mean

    def train_step(self):
        x = self.source_sampler.sample(self.config.batch_size).to(self.config.device)
        y = self.target_sampler.sample(self.config.batch_size).to(self.config.device)
        loss = self.compute_loss(x, y)
        self.opt_both.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.f_theta.parameters(), max_norm=self.config.grad_clip)
        torch.nn.utils.clip_grad_norm_(self.f_phi.parameters(), max_norm=self.config.grad_clip)
        self.opt_both.step()
        self.ema_update(self.f_theta, self.f_theta_ema)
        self.current_step += 1

    def train(self, viz_callback=None):
        print(f"Starting training name = {self.experiment_name}")
        print("-" * 60)

        while True:
            self.train_step()
            viz_callback(self)
            if (self.current_step >= self.config.epoch):
                break

        return 0


    def score_y_given_x(self, y, x):
        y = y.detach().requires_grad_(True)

        ft = self.f_theta_ema(y).sum()
        (gy,) = torch.autograd.grad(ft, y, retain_graph=False, create_graph=False)
        return (gy - (y - x)) / self.config.eps(self.current_step)

    def sample_pi_given_x(self, x, n=200):
        y = x.unsqueeze(1) + 0.0 * (self.config.eps(self.current_step) ** 0.5) * torch.randn(x.shape[0], n, x.shape[1], device=self.config.device)
        for _ in range(self.config.lmc_steps):
            y = y.detach()
            sc = self.score_y_given_x(y, x.unsqueeze(1))
            with torch.no_grad():
                y = y + self.config.lmc_step_size * sc + math.sqrt(2*self.config.lmc_step_size) * torch.randn_like(y)
        return y.detach()


    def save_checkpoint(self, filename):
        checkpoint = {
            'f_theta_state_dict': self.f_theta_ema.state_dict(),
            'f_phi_state_dict': self.f_phi.state_dict(),
            'epoch': self.current_step,
            'eps': self.config.eps(self.current_step),
        }
        torch.save(checkpoint, filename)

In [10]:
def mmd(x, y):
    Kxx = pairwise_distances(x, x)
    Kyy = pairwise_distances(y, y)
    Kxy = pairwise_distances(x, y)
    m, n = x.shape[0], y.shape[0]
    A = np.sum(Kxx - np.diag(np.diagonal(Kxx)))
    B = np.sum(Kyy - np.diag(np.diagonal(Kyy)))
    C = np.sum(Kxy)
    return -0.5 / (m * (m - 1)) * A - 0.5 / (n * (n - 1)) * B + 1 / (m * n) * C


def evaluate(trainer, n_samples=None):
    trainer.f_theta_ema.eval()

    n_src = start_data_scaled.shape[0] if n_samples is None else n_samples
    n_tgt = end_data_scaled.shape[0]   if n_samples is None else n_samples

    X = X_sampler.sample(n_src).to(DEVICE)

    Y_pred = trainer.sample_pi_given_x(X, n=1).squeeze(1)

    Y_pred_raw = Y_pred.cpu().numpy() * constant_scale
    Y_raw      = end_data_scaled * constant_scale
    mmd1 = mmd(Y_pred_raw, Y_raw)

    t_val = (DAY_EVAL - DAY_START) / (DAY_END - DAY_START)
    eps   = trainer.config.eps(trainer.current_step)

    with torch.no_grad():
        noise = torch.randn_like(X)
        X_mid = (1 - t_val) * X + t_val * Y_pred \
                + math.sqrt(eps * t_val * (1 - t_val)) * noise
        X_mid_raw = X_mid.cpu().numpy() * constant_scale

    mmd2 = mmd(X_mid_raw, eval_data_raw.numpy())

    return mmd1, mmd2

In [ ]:
config = EOTConfig(
    eps               = lambda step: EPSILON,
    batch_size        = BATCH_SIZE,
    device            = DEVICE,
    K                 = K,
    epoch             = EPOCH,
    lmc_steps         = LMC_STEPS,
    lmc_step_size     = LMC_STEP_SIZE,
    ema_momentum      = 0.0,
    max_diff_exp_clip = 25,
    grad_clip         = 1e7,
)

trainer = EOTTrainer(
    config         = config,
    source_sampler = X_sampler,
    target_sampler = Y_sampler,
    model_theta    = MLP(input_dim=DIM, hiddens=hiddens[DIM], output_dim=1).to(DEVICE),
    model_phi      = MLP(input_dim=DIM, hiddens=hiddens[DIM], output_dim=1).to(DEVICE),
    name           = EXP_NAME,
)

LR_   = LR
BETAS = (0.9, 0.999)
WD    = 1e-4

trainer.opt_both = torch.optim.AdamW(
    list(trainer.f_theta.parameters()) + list(trainer.f_phi.parameters()),
    lr=LR_, betas=BETAS, weight_decay=WD,
)

trainer.best_mmd_mid = float('inf')

import types

def update_best(self, mmd_mid):
    if mmd_mid < self.best_mmd_mid:
        self.best_mmd_mid = mmd_mid
        self.save_checkpoint(os.path.join(OUTPUT_PATH, 'best.pt'))

trainer.update_best = types.MethodType(update_best, trainer)

print(f'f_theta params: {sum(p.numel() for p in trainer.f_theta.parameters()):,}')
print(f'f_phi   params: {sum(p.numel() for p in trainer.f_phi.parameters()):,}')

In [ ]:
def viz_callback(trainer):
    if trainer.current_step % EVAL_EVERY != 0:
        return
    metrics = evaluate(trainer)
    trainer.update_best(metrics[1])

start_time = time.time()
trainer.train(viz_callback=viz_callback)
print(f'Training time: {time.time() - start_time:.1f}s')

metrics = evaluate(trainer)